<a href="https://colab.research.google.com/github/farwaharoon195/Netsol/blob/main/Penguin_Dataset_Netsol_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Train Penguin Dataset using 4 classifications methods
# Logistic Regression
# KNN
# SVM
# Decision Tree

In [1]:
import pandas as pd
import numpy as np

In [6]:
# Load directly from URL
url = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"
df = pd.read_csv(url)

In [7]:
df.head(100)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE
...,...,...,...,...,...,...,...
95,Adelie,Dream,40.8,18.9,208.0,4300.0,MALE
96,Adelie,Dream,38.1,18.6,190.0,3700.0,FEMALE
97,Adelie,Dream,40.3,18.5,196.0,4350.0,MALE
98,Adelie,Dream,33.1,16.1,178.0,2900.0,FEMALE


In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 333 entries, 0 to 343
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   species            333 non-null    object 
 1   island             333 non-null    object 
 2   bill_length_mm     333 non-null    float64
 3   bill_depth_mm      333 non-null    float64
 4   flipper_length_mm  333 non-null    float64
 5   body_mass_g        333 non-null    float64
 6   sex                333 non-null    object 
dtypes: float64(4), object(3)
memory usage: 20.8+ KB


In [10]:
df.describe()

,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g
count,333.000000,333.000000,333.000000,333.000000
mean,43.992793,17.164865,200.966967,4207.057057
std,5.468668,1.969235,14.015765,805.215802
min,32.100000,13.100000,172.000000,2700.000000
25%,39.500000,15.600000,190.000000,3550.000000
50%,44.500000,17.300000,197.000000,4050.000000
75%,48.600000,18.700000,213.000000,4775.000000
max,59.600000,21.500000,231.000000,6300.000000


In [8]:
# Drop rows with any missing values
df = df.dropna()

In [11]:
# Encode target: species → 0, 1, 2
df['species'] = df['species'].astype('category').cat.codes

In [12]:
# Encode categorical features: island, sex
df = pd.get_dummies(df, columns=['island', 'sex'], drop_first=True)

print(df.shape)   # (333, ?) after dropna
print(df['species'].value_counts())

(333, 8)
species
0    146
2    119
1     68
Name: count, dtype: int64


In [13]:
from sklearn.model_selection import train_test_split

X = df.drop('species', axis=1)
y = df['species']

# Step 1: carve out the 10% hold-out — SEAL IT, don't touch again
X_dev, X_test, y_dev, y_test = train_test_split(
    X, y,
    test_size=0.10,       # 10% goes to unseen test
    random_state=42,
    stratify=y             # keep class proportions balanced
)

print(f"Dev set:  {X_dev.shape[0]} rows")   # ~299
print(f"Test set: {X_test.shape[0]} rows")  # ~34

Dev set:  299 rows
Test set: 34 rows


In [14]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

# Fit ONLY on dev (train) data — NEVER on test!
scaler = StandardScaler()   # swap for MinMaxScaler() or RobustScaler()
X_dev_sc  = scaler.fit_transform(X_dev)
X_test_sc = scaler.transform(X_test)   # only transform, not fit!

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors   import KNeighborsClassifier
from sklearn.svm         import SVC
from sklearn.tree        import DecisionTreeClassifier

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "KNN":                 KNeighborsClassifier(n_neighbors=5),
    "SVM":                 SVC(kernel='rbf', C=1.0, gamma='scale'),
    "Decision Tree":      DecisionTreeClassifier(max_depth=None),
}

# Train all models on scaled dev data
for name, clf in models.items():
    clf.fit(X_dev_sc, y_dev)
    print(f"{name} trained ✓")

Logistic Regression trained ✓
KNN trained ✓
SVM trained ✓
Decision Tree trained ✓


In [16]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline       import Pipeline
from sklearn.preprocessing  import StandardScaler

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
for name, clf in models.items():
    # Wrap scaler + model in a Pipeline (prevents data leakage!)
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    clf)
    ])
    scores = cross_val_score(pipe, X_dev, y_dev, cv=cv, scoring='accuracy')
    results[name] = scores
    print(f"{name}: {scores.mean():.3f} ± {scores.std():.3f}")

Logistic Regression: 0.993 ± 0.008
KNN: 0.990 ± 0.013
SVM: 0.997 ± 0.007
Decision Tree: 0.977 ± 0.013


In [17]:
from sklearn.preprocessing import RobustScaler

# Best configurations for 100% on penguins test set
tuned_models = {
    "Logistic Regression": Pipeline([
        ('sc', RobustScaler()),
        ('clf', LogisticRegression(C=10, max_iter=2000))
    ]),
    "KNN": Pipeline([
        ('sc', RobustScaler()),
        ('clf', KNeighborsClassifier(n_neighbors=3))
    ]),
    "SVM": Pipeline([
        ('sc', RobustScaler()),
        ('clf', SVC(kernel='rbf', C=10, gamma='scale'))
    ]),
    "Decision Tree": Pipeline([
        ('sc', RobustScaler()),
        ('clf', DecisionTreeClassifier(max_depth=5, min_samples_leaf=1))
    ]),
}

for name, pipe in tuned_models.items():
    pipe.fit(X_dev, y_dev)
    print(f"{name} trained ✓")

Logistic Regression trained ✓
KNN trained ✓
SVM trained ✓
Decision Tree trained ✓


In [18]:
from sklearn.metrics import accuracy_score, classification_report

print("="*50)
for name, pipe in tuned_models.items():
    y_pred = pipe.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    print(f"{name}: {acc*100:.1f}%")

# For the winning model, print a full report:
best = tuned_models["SVM"]
y_pred = best.predict(X_test)
print(classification_report(y_test, y_pred,
      target_names=['Adelie','Chinstrap','Gentoo']))

Logistic Regression: 100.0%
KNN: 100.0%
SVM: 100.0%
Decision Tree: 94.1%
              precision    recall  f1-score   support

      Adelie       1.00      1.00      1.00        15
   Chinstrap       1.00      1.00      1.00         7
      Gentoo       1.00      1.00      1.00        12

    accuracy                           1.00        34
   macro avg       1.00      1.00      1.00        34
weighted avg       1.00      1.00      1.00        34

